In [0]:
import logging
import yaml
import requests
from tenacity import retry, stop_after_attempt, wait_exponential

logger = logging.getLogger("eia_ingestion")
logger.setLevel(logging.INFO)

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("config_path", "")

environment = dbutils.widgets.get("environment")
CONFIG_PATH = dbutils.widgets.get("config_path")

try:
    with open(CONFIG_PATH) as f:
        full_config = yaml.safe_load(f)

    config = full_config[environment]
    volume_path = config["bronze_volumes_path"]
    catalog = config["catalog"]
    bronze_schema = config["bronze_schema"]

    logger.info(f"[{environment}] Config loaded. Volume path: {volume_path}")

except Exception as e:
    logger.error(f"Failed to load config from {CONFIG_PATH} for environment '{environment}': {e}")
    raise

In [0]:
from datetime import date, timedelta

CONSUMPTION_URL = "https://api.eia.gov/v2/petroleum/cons/wpsup/data/"
PRICES_GND_URL = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"
PRICES_SPT_URL = "https://api.eia.gov/v2/petroleum/pri/spt/data/"

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=8))
def fetch_page(url, offset, start, end, length=5000, extra_params=None):
    params = {
        "frequency": "weekly",
        "data[0]": "value",
        "start": start,
        "end": end,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": offset,
        "length": length,
        "api_key": API_KEY,
    }
    if extra_params:
        params.update(extra_params)
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response


def fetch_all(url, start, end, extra_params=None, page_size=5000):
    all_data = []
    offset = 0
    total = None

    while total is None or offset < total:
        response = fetch_page(url, offset=offset, start=start, end=end, length=page_size, extra_params=extra_params)
        body = response.json()["response"]

        if total is None:
            total = int(body["total"])
            logger.info(f"Total rows available: {total}")

        all_data.extend(body["data"])
        offset += page_size
        logger.info(f"Fetched {len(all_data)}/{total} rows so far")

    return all_data


END_DATE = date.today().isoformat()
START_DATE = (date.today() - timedelta(days=30)).isoformat()

In [0]:
ROUTE_URLS = {
    "gnd": PRICES_GND_URL,
    "spt": PRICES_SPT_URL,
}

mapping_table = f"{catalog}.{bronze_schema}.product_price_mapping"

mapping_rows = (
    spark.table(mapping_table)
    .filter("is_active = true")
    .select("consumption_product_code", "price_route", "price_product_code")
    .collect()
)

product_price_mapping = {}
for row in mapping_rows:
    product_price_mapping.setdefault(row["consumption_product_code"], []).append(
        {"route": row["price_route"], "price_code": row["price_product_code"]}
    )

logger.info(f"Loaded {len(product_price_mapping)} active product mappings from {mapping_table}.")

consumption_data = fetch_all(CONSUMPTION_URL, start=START_DATE, end=END_DATE)

if not consumption_data:
    logger.warning("Consumption data is empty — prices will not be filtered by anything.")
    prices_data = []
else:
    consumption_codes = {row["product"] for row in consumption_data if row.get("product")}
    logger.info(f"Unique product codes in consumption: {sorted(consumption_codes)}")

    unmapped_codes = consumption_codes - product_price_mapping.keys()
    if unmapped_codes:
        logger.warning(f"No active price mapping for these consumption codes (skipped): {sorted(unmapped_codes)}")

    codes_by_route = {}
    for code in consumption_codes:
        for mapping in product_price_mapping.get(code, []):
            route_url = ROUTE_URLS[mapping["route"]]
            codes_by_route.setdefault(route_url, set()).add(mapping["price_code"])

    prices_data = []
    for route_url, price_codes in codes_by_route.items():
        extra_params = {"facets[product][]": sorted(price_codes)}
        route_data = fetch_all(route_url, start=START_DATE, end=END_DATE, extra_params=extra_params)
        logger.info(f"Fetched {len(route_data)} price rows from {route_url} for codes {sorted(price_codes)}")
        prices_data.extend(route_data)

    logger.info(f"Total price rows fetched: {len(prices_data)}")

In [0]:
import json

def save_to_volume(data, file_name):
    file_path = volume_path + file_name
    with open(file_path, "w") as f:
        json.dump(data, f)
    logger.info(f"Saved {len(data)} records to {file_path}")


save_to_volume(consumption_data, "petroleum_raw.json")
save_to_volume(prices_data, "petroleum_prices_raw.json")